### RDD - Transformations

In [1]:
## Transformation 

# map 
# flatMap
# filter
# distinct 
# union 
# intersection
# groupBy
# groupByKey
# reduceBykey 

# Actions 
# collect()
# take()
# count()
# reduce() 
# first()
# takeSample(withReplacement , num , seed )
# saveAsTextFile()
# foreach(func)




In [1]:
import os

# Set the correct Java Home
os.environ['JAVA_HOME'] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

# Set the Spark Home
os.environ['SPARK_HOME'] = "/opt/homebrew/opt/apache-spark/libexec"

In [2]:
from pyspark.sql import SparkSession

# The data to be used
data = [1, 2, 3, 4]

# Create a SparkSession
spark = SparkSession.builder.appName("RDD").getOrCreate()

# Get the SparkContext from the SparkSession
sc = spark.sparkContext

# Create an RDD using the SparkContext's parallelize method
rdd = sc.parallelize(data)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/02 12:27:00 WARN Utils: Your hostname, mukeshs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.221 instead (on interface en0)
26/02/02 12:27:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/02 12:27:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
# wordCount.py 

from pyspark.sql import SparkSession 

#spark = SparkSession.builder.appName("Test").getOrCreate()

sc = spark.sparkContext

# Step 1 :  Load Data (E)
# Sources : txt , csv , tsv , parquet , ORC , Avro , JSON , XML 
rdd = sc.textFile("s3://")


# Step 2: Transformations (T)
words = rdd.flatMap(lambda line : line.split(" "))
pairs = words.map(lambda word: (word,1))

word_counts = pairs.reduceByKey(lambda a , b: a+b)


# Step 3 : Action (L)

result = word_counts.collect() # saveasTextFile (s3 save)


# spark-submit .... wordCount.py 


"""
    Driver : Run the above code 
    Job    : Triggerred by collect()
    Stages :  Two stages :
            Stage 1 : flatMap , map ( narrow transformation )
            Stage 2 : reduceByKey (wide transformation + shuffling )
    Tasks : Each stage is split into 
            If rdd 8 partitions ---> 8 tasks per stage 

Driver 
---------Job ( triggerred by collect )
         ------------Stage 1 (flatmap , map)
                       ------------Tasks (1...8)
         ------------Stage 2 (reduceByKey , )                          

"""


Py4JJavaError: An error occurred while calling o54.partitions.
: org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3581)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.hadoop.mapred.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:276)
	at org.apache.hadoop.mapred.FileInputFormat.listStatus(FileInputFormat.java:245)
	at org.apache.hadoop.mapred.FileInputFormat.getSplits(FileInputFormat.java:334)
	at org.apache.spark.rdd.HadoopRDD.getPartitions(HadoopRDD.scala:233)
	at org.apache.spark.rdd.RDD.$anonfun$p

In [5]:
# Schema 

# inferSchema = True 

from pyspark.sql.types import StructType , StructField , StringType, IntegerType , DateType

# define the schema 

# ('mukesh','suyal',40,'11/09/2025','IT')

myschema = StructType([
               StructField("firstName",StringType(),False),
               StructField("lastName",StringType(),False),
               StructField("age",IntegerType(),False),
               StructField("Date",DateType(),False),
               StructField("Department",StringType(),False)
])


myschema = "firstname STRING , lastName STRING "
df = spark.read.csv("path to csv file",header=True,schema=myschema)

df.printSchema()

df.show()





25/10/12 14:51:57 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: path to csv file.
java.io.FileNotFoundException: File path to csv file does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/Users/mukesh/Desktop/Trainings/nodeB/dataeng/PYSPARK_CODE/path to csv file. SQLSTATE: 42K03

In [4]:
from pyspark.sql import SparkSession

# Example for Delta Lake
# Make sure to use versions compatible with your Spark installation
spark = SparkSession.builder \
    .appName("DeltaExample") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .getOrCreate()

25/09/08 09:07:29 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
# map 
data1 = [1,2,3,4]

rdd = sc.parallelize(data1)

result = rdd.map(lambda x : x*2)


# filter 
data2 = [1,2,3,4,5,6,7,8]
rdd = sc.parallelize(data2)

filter_rdd = rdd.filter(lambda x : x % 2 == 0)


#flatMap 

data3 = ["hello   world " , "spark rddd"]

rdd = sc.parallelize(data3)

flat_rdd = rdd.flatMap(lambda line : line.split()) 


# distinct 

data4 = [1, 2, 2, 3, 4, 4]

rdd = sc.parallelize(data4)

distinct_rdd = rdd.distinct()


# union & intersection 

data1 = [1,2,3,4,5,9,10 ]
data2 = [9,10]

rdd1 = sc.parallelize(data1)
rdd2 = sc.parallelize(data2)


union_rdd = rdd1.union(rdd2)

intersection = rdd1.intersection(rdd2)

# group BY 

data = [1,2,3,4,5]

rdd = sc.parallelize(data)

grouped_rdd = rdd.groupBy(lambda x : x % 2)


"""
Output : 
    0 : [2,4]
    1 : [1,3,5]
"""



'\nOutput : \n    0 : [2,4]\n    1 : [1,3,5]\n'

25/09/09 20:53:21 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 683704 ms exceeds timeout 120000 ms
25/09/09 20:53:21 WARN SparkContext: Killing executors is not supported by current scheduler.
25/09/09 20:53:24 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [ ]:
# reduceByKey
# groupByKey 


# reduceByKey 

["apple banana" ,"banana apple mango " , "mango banana "] # rdd 

["apple", "banana" ,"banana", "apple", "mango" , "mango" "banana"] # flatRDD

pairs = [("apple",1) , ("banana",1) , ("banana",1) , ("apple",1), ("mango",1) , ("mango",1) , ("banana",1)] # PairRDD


result = pairs.reduceByKey(lambda a , b : a +b )

("apple",1). 1+1 = 2
("apple",1), 

("apple",1)
("apple",1),


result = pairs.groupByKey(lambda a , b : a +b )

reduceByKey : It perfoms local aggregation before shuffling 
groupByKey : It shuffles all vcalues before aggration




data1 = [("s1","Alice"),("s2","mukesh")]
data2 = [("s1",85),("s2",90)]


rdd1 = sc.parallelize(data1)
rdd2 = sc.parallelize(data2)


joined = rdd1.join()


In [8]:
data1 = [ ("a",1) , ("b",2)]
data2 = [ ("a" , 3) , ("a" , 4) , ("b" , 5) ]


rdd1 = sc.parallelize(data1)
rdd2 = sc.parallelize(data2)


cogroup  = rdd1.cogroup(rdd2)


 ## "a" , [[1],[2,4]] , "b" , [[2],[5]]

# Partitioning  Bucketing
# 
# 
#  

SyntaxError: invalid syntax (4217404786.py, line 1)

In [1]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder.appName("Hello").getOrCreate()

ConnectionRefusedError: [Errno 61] Connection refused

In [14]:
!which pyspark

/Users/mukesh/Desktop/Trainings/nodeB/venv/bin/pyspark


In [15]:
!brew --prefix apache-spark

/opt/homebrew/opt/apache-spark


In [16]:
!/usr/libexec/java_home

/Library/Java/JavaVirtualMachines/temurin-23.jdk/Contents/Home


In [17]:
!export SPARK_HOME="/Users/mukesh/Desktop/Trainings/nodeB/venv/bin/pyspark"
!export JAVA_HOME="/Library/Java/JavaVirtualMachines/temurin-23.jdk/Contents/Home"
!export PATH="/Users/mukesh/Desktop/Trainings/nodeB/venv/bin/pyspark"

In [3]:
!pip freeze | grep pyspark

pyspark==4.0.0
